# Root finding and optimization — code from the lecture

Every code example from the *Root finding and optimization* chapter of the lecture
notes, in the order it appears there: <https://dse.iskh.me/solvers>

Run the cells top to bottom — later ones use names defined earlier.

## Bisection method

In [ ]:
def bisection(f,a=0,b=1,tol=1e-6,maxiter=100,callback=None):
  '''Bisection method for solving equation f(x)=0
  on the interval [a,b], with given tolerance and number of iterations.
  Callback function is invoked at each iteration if given.
  '''
  if f(a)*f(b)>0:
    raise ValueError('Function has the same sign at the bounds')
  for i in range(maxiter):
    err = abs(b-a)
    if err<tol: break
    x = (a+b)/2
    if callback != None: callback(err=err,x=x,iter=i,a=a,b=b)
    a,b = (x,b) if f(a)*f(x)>0 else (a,x)
  else:
    raise RuntimeError('Failed to converge in %d iterations'%maxiter)
  return x

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [9, 6]

f = lambda x: -4*x**3+5*x+1
a,b = -3,-.5  # upper and lower limits
xd = np.linspace(a,b,1000)  # x grid
ylim = [-5,25]  # vertical zoom: what matters is the sign, the function runs off the top
def plot_step(a,b,x,iter,**kwargs):
    plot_step.counter += 1
    if iter<7:
        lo,hi = (x,b) if f(a)*f(x)>0 else (a,x)  # the half that is kept
        plt.plot(xd,f(xd),c='red')  # plot the function
        plt.plot([xd[0],xd[-1]],[0,0],c='black')  # plot zero line
        plt.fill_between([a,b],*ylim,color='grey',alpha=0.10)  # current bracket
        plt.fill_between([lo,hi],*ylim,color='green',alpha=0.15)  # the half to keep
        plt.plot([a,a],ylim,c='grey')  # plot lower bound
        plt.plot([b,b],ylim,c='grey')  # plot upper bound
        plt.plot([x,x],ylim,c='green')  # plot the midpoint
        plt.scatter([a,b,x],[f(a),f(b),f(x)],c=['grey','grey','green'],zorder=3)
        plt.title('Iteration %d: [%1.4f,%1.4f], f(x)=%+1.4f, keep [%1.4f,%1.4f]'
                  %(iter+1,a,b,f(x),lo,hi))
        plt.xlim(xd[0],xd[-1])
        plt.ylim(ylim)
        plt.show()
plot_step.counter = 0  # new public attribute
bisection(f,a,b,callback=plot_step)
print('Converged in %d steps'%plot_step.counter)

## Derivation of Newton step

In [ ]:
def newton(fun,grad,x0,tol=1e-6,maxiter=100,callback=None):
    '''Newton method for solving equation f(x)=0
    with given tolerance and number of iterations.
    Callback function is invoked at each iteration if given.
    '''
    for i in range(maxiter):
        x1 = x0 - fun(x0)/grad(x0)
        err = abs(x1-x0)
        if callback != None: callback(err=err,x0=x0,x1=x1,iter=i)
        if err<tol: break
        x0 = x1
    else:
        raise RuntimeError('Failed to converge in %d iterations'%maxiter)
    return (x0+x1)/2

In [ ]:
f = lambda x: -4*x**3+5*x+1
g = lambda x: -12*x**2+5
a,b = -3,-.5  # upper and lower limits
xd = np.linspace(a,b,1000)  # x grid
def plot_step(x0,x1,iter,**kwargs):
    plot_step.counter += 1
    if iter<5:
        plt.plot(xd,f(xd),c='red')  # plot the function
        plt.plot([a,b],[0,0],c='black')  # plot zero line
        ylim = [min(f(b),0),f(a)]
        plt.plot([x0,x0],ylim,c='grey') # plot x0
        l = lambda z: g(x0)*(z - x1)
        plt.plot(xd,l(xd),c='green')  # plot the tangent line
        plt.ylim(bottom=10*f(b))
        plt.title('Iteration %d'%(iter+1))
        plt.show()
plot_step.counter = 0  # new public attribute
newton(f,g,x0=-2.5,callback=plot_step)
print('Converged in %d steps'%plot_step.counter)

## Rates of convergence and complexity

In [ ]:
xstar = -1.0  # f(-1) = 4-5+1 = 0, the exact root in the bracket

def print_err(iter,err,**kwargs):
    '''Report the error at each iteration of either solver.
    Bisection passes the midpoint as x, Newton–Raphson the new point as x1.
    '''
    x = kwargs['x'] if 'x' in kwargs else kwargs['x1']
    print('%3d:  x = %+1.14f   |x-x*| = %1.3e   reported err = %1.3e'
          %(iter,x,abs(x-xstar),err))

print('Bisection on [-3,-0.5]')
bisection(f,-3,-.5,callback=print_err)
print('\nNewton–Raphson from x0=-2.5')
newton(f,g,x0=-2.5,callback=print_err)

In [ ]:
def collect(store):
    '''Make a callback that appends (true error, reported error) to the given list'''
    def cb(iter,err,**kwargs):
        x = kwargs['x'] if 'x' in kwargs else kwargs['x1']
        store.append((abs(x-xstar),err))
    return cb

eb, en = [], []
bisection(f,-3,-.5,callback=collect(eb))
newton(f,g,x0=-2.5,callback=collect(en))
eb, en = np.array(eb), np.array(en)

plt.semilogy(np.arange(1,len(eb)+1),eb[:,0],'o',c='blue',label=r'bisection $|x_k-x^\star|$')
plt.semilogy(np.arange(1,len(eb)+1),eb[:,1],'-',c='blue',alpha=.5,label='bisection bracket width')
plt.semilogy(np.arange(1,len(en)+1),en[:,0],'o-',c='red',label=r'Newton–Raphson $|x_k-x^\star|$')
plt.xlabel('iteration $k$')
plt.legend()
plt.grid(True,which='both',alpha=.3)
plt.show()

## Multiple solutions

In [ ]:
def newton_pic(f,g,x0,a=0,b=1,**kwargs):
    '''Illustrate the Newton method iterations on the interval [a,b]'''
    xd = np.linspace(a,b,1000)
    plt.plot(xd,f(xd),c='red')       # the function
    plt.plot([a,b],[0,0],c='black')  # zero line
    def plot_step(**kw):
        plot_step.counter += 1
        z0,z1 = kw['x0'],kw['x1']
        plt.plot([z0,z0],[0,f(z0)],c='green')  # from the axis up to the function
        plt.plot([z0,z1],[f(z0),0],c='green')  # tangent line down to the axis
    plot_step.counter = 0
    try:
        xs = newton(f,g,x0,callback=plot_step,**kwargs)
        plt.title('Started at %1.3f, converged to %1.5f in %d steps'%(x0,xs,plot_step.counter))
    except RuntimeError:
        plt.title('Started at %1.3f, failed to converge in %d iterations'%(x0,plot_step.counter))
    plt.xlim((a,b))
    plt.show()

f = lambda x: -4*x**3+5*x+1  # function
g = lambda x: -12*x**2+5     # derivative
for x0 in [-0.565,-0.58,-0.595]:
    newton_pic(f,g,x0,a=-3,b=1.5)

## Divergence and the domain of attraction

In [ ]:
f = lambda x: np.arctan(x)
g = lambda x: 1/(1+x**2)
newton_pic(f,g,x0=1.25,a=-20,b=20)            # inside the domain of attraction
newton_pic(f,g,x0=1.5,a=-20,b=20,maxiter=8)   # outside it

## Cycles

In [ ]:
f = lambda x: -4*x**3+5*x+1  # function
g = lambda x: -12*x**2+5     # derivative
h = lambda x: -24*x          # second derivative

ns = lambda x: x - f(x)/g(x)          # the Newton step itself
ds = lambda x: f(x)*h(x)/g(x)**2      # its derivative
f2 = lambda x: ns(ns(x)) - x          # two Newton steps return to the start
g2 = lambda x: ds(ns(x))*ds(x) - 1    # derivative of the above

x0 = newton(f2,g2,x0=-0.56,tol=1e-16)  # find the cycling starting point
print('To cycle start with x0 = %1.16f'%x0)

newton_pic(f,g,x0,a=-1.5,b=1.5,maxiter=15)

## Function domain and differentiability

In [ ]:
f = lambda x: np.log(x)
g = lambda x: 1/x
newton_pic(f,g,x0=2.9,a=0.001,b=3)

## Suboptimal performance

In [ ]:
f = lambda x: x**9   # a very special case: root of multiplicity 9
g = lambda x: 9*x**8
newton_pic(f,g,x0=1.0,a=-1.5,b=1.5)

def print_err(**kwargs):
    if kwargs['iter'] % 5 == 0:
        print('{:4d}:  x = {:17.14f}  err = {:8.6e}'.format(kwargs['iter'],kwargs['x1'],kwargs['err']))
newton(f,g,x0=1.0,callback=print_err)

## An objective function with many critical points

In [ ]:
def quad(x,y,c,d,A,B):
    '''Exponent q of a plain Gaussian bump, with its gradient and Hessian'''
    q   = (x-c)**2/A + (y-d)**2/B
    dq  = [2*(x-c)/A, 2*(y-d)/B]
    d2q = [[2/A+0*x, 0*x],[0*x, 2/B+0*x]]
    return q,dq,d2q

def quad_banana(x,y):
    '''Exponent q of the curved ridge, with its gradient and Hessian'''
    u = x - 0.55
    v = y - 0.63 + 2.5*u**2
    q   = u**2/0.055 + v**2/0.0025
    dq  = [2*u/0.055 + 4000*u*v, 800*v]
    d2q = [[2/0.055 + 4000*v + 20000*u**2, 4000*u],[4000*u, 800+0*x]]
    return q,dq,d2q

def bumps(x,y):
    '''The four terms of F: amplitude a, exponent q, and the derivatives of q'''
    return [(1.00,) + quad(x,y,0.22,0.27,0.018,0.030),
            (0.94,) + quad(x,y,0.73,0.25,0.080,0.012),
            (0.78,) + quad_banana(x,y),
            (1.20,) + quad(x,y,0.55,0.48,0.005,0.010)]

def F(x,y):
    return sum(a*np.exp(-q) for a,q,dq,d2q in bumps(x,y))

In [ ]:
plt.rcParams['figure.figsize'] = [9, 6]
def contour_plot(fun,levels=30,xlim=(0,1.2),ylim=(0,0.72),npoints=200,ax=None,clip=None):
    '''Contour plot of a function of two variables.
    With clip=p the levels are symmetric around zero and cut at the p-th percentile
    of |Z|, which keeps a few extreme values from swamping the picture.
    '''
    X,Y = np.meshgrid(np.linspace(*xlim,npoints),np.linspace(*ylim,npoints))
    Z = fun(X,Y)
    if clip is None:
        lv = np.linspace(Z.min(),Z.max(),levels)
        lv = np.concatenate([np.exp(np.linspace(np.log(Z.min()),np.log(0.2),levels//2)),np.linspace(0.3,Z.max(),levels//2)])
    else:
        c = np.percentile(np.abs(Z),clip)
        lv = np.linspace(-c,c,levels)
    if ax is None:
        fig, ax = plt.subplots()
    ax.contour(X,Y,Z,levels=lv)
    ax.set_aspect('equal','box')
    ax.set_xlim(*xlim)  # fix the window: paths drawn on top may leave it
    ax.set_ylim(*ylim)
    return ax

contour_plot(F)
plt.show()

## Gradient and Hessian

In [ ]:
def G(x,y):
    '''Gradient of F: grad(a exp(-q)) = -a exp(-q) grad(q)'''
    out = [0,0]
    for a,q,dq,d2q in bumps(x,y):
        g = a*np.exp(-q)
        for k in range(2):
            out[k] = out[k] - g*dq[k]
    return out

def H(x,y):
    '''Hessian of F: H(a exp(-q)) = a exp(-q) [grad(q) grad(q)' - H(q)]'''
    out = [[0,0],[0,0]]
    for a,q,dq,d2q in bumps(x,y):
        g = a*np.exp(-q)
        for k in range(2):
            for j in range(2):
                out[k][j] = out[k][j] + g*(dq[k]*dq[j] - d2q[k][j])
    return out

In [ ]:
eps = 1e-6
for x,y in [(0.30,0.70),(0.55,0.50),(0.80,0.20)]:
    Gn = [(F(x+eps,y)-F(x-eps,y))/(2*eps), (F(x,y+eps)-F(x,y-eps))/(2*eps)]
    Hn = [[(G(x+eps,y)[k]-G(x-eps,y)[k])/(2*eps) for k in range(2)],
          [(G(x,y+eps)[k]-G(x,y-eps)[k])/(2*eps) for k in range(2)]]
    print('at (%4.2f,%4.2f):  max |G-Gnum| = %.2e   max |H-Hnum| = %.2e'
          % (x,y,np.amax(np.abs(np.array(G(x,y))-np.array(Gn))),
                 np.amax(np.abs(np.array(H(x,y))-np.array(Hn).T))))

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(11,5))
for k in range(2):
    contour_plot(lambda x,y: G(x,y)[k],ax=axs[k],clip=95,levels=20)
    axs[k].set_title('$G_%d(x,y)$'%(k+1))
plt.show()

fig, axs = plt.subplots(1,2,figsize=(11,10))
k=0
for j in range(2):
    contour_plot(lambda x,y: H(x,y)[k][j],ax=axs[j],clip=95,levels=20)
    axs[j].set_title('$H_{%d%d}(x,y)$'%(k+1,j+1))
plt.show()

fig, axs = plt.subplots(1,2,figsize=(11,10))
k=1
for j in range(2):
    contour_plot(lambda x,y: H(x,y)[k][j],ax=axs[j],clip=95,levels=20)
    axs[j].set_title('$H_{%d%d}(x,y)$'%(k+1,j+1))
plt.show()

## Multivariate Newton solver

In [ ]:
def newton2(fun,grad,x0,tol=1e-6,maxiter=100,callback=None):
    '''Newton method for solving a system of equations fun(x)=0,
    where x is a vector of 2 elements and grad is the Jacobian.
    Callback function is invoked at each iteration if given.
    '''
    # conversion to array function of array argument
    npfun  = lambda x: np.asarray(fun(x[0],x[1]))
    npgrad = lambda x: np.asarray(grad(x[0],x[1]))
    x0 = np.asarray(x0,dtype=float)
    for i in range(maxiter):
        x1 = x0 - np.linalg.solve(npgrad(x0),npfun(x0))  # matrix version
        err = np.amax(np.abs(x1-x0))  # vector sup norm
        if callback != None: callback(iter=i,err=err,x0=x0,x1=x1)
        if err<tol: break
        x0 = x1
    else:
        raise RuntimeError('Failed to converge in %d iterations'%maxiter)
    return x1

In [ ]:
def newton_path(x0,**kwargs):
    '''Solve G(x)=0 from x0, recording the sequence of Newton steps'''
    path = [np.asarray(x0,dtype=float)]
    xs = newton2(G,H,x0,callback=lambda **kw: path.append(kw['x1']),**kwargs)
    return xs, np.array(path)

def plot_newton_path(x0,**kwargs):
    '''Plot the Newton iterations on top of the contours of F'''
    xs, path = newton_path(x0,**kwargs)
    ax = contour_plot(F)
    ax.plot(path[:,0],path[:,1],c='r',marker='.')  # the path
    ax.scatter(*path[0],c='r',marker='o')          # starting point
    ax.scatter(*path[-1],c='b',marker='*',s=150,zorder=3)  # solution
    ax.set_title('Started at (%1.2f,%1.2f), converged in %d steps'%(*path[0],len(path)-1))
    plt.show()

In [ ]:
plot_newton_path([0.55,0.45])

In [ ]:
plot_newton_path([0.55,0.40])

In [ ]:
def newton_trace(x0,solver=newton2,**kwargs):
    '''Newton path from x0, with the reason the solver stopped'''
    path = [np.asarray(x0,dtype=float)]
    try:
        solver(G,H,x0,callback=lambda **kw: path.append(kw['x1']),**kwargs)
        status = 'converged'
    except np.linalg.LinAlgError:
        status = 'singular Hessian'
    except RuntimeError as e:
        status = 'no ascent direction' if 'ascent' in str(e) else 'maxiter reached'
    return np.array(path), status

center, side = np.array([0.6,0.43]), 0.05  # the small square of starting points
starts = center + np.random.default_rng(44).uniform(-side/2,side/2,size=(10,2))
colors = plt.cm.autumn(np.linspace(0,0.85,len(starts)))  # slightly different colors

def plot_newton_paths(solver=newton2,starts=starts,colors=colors,**kwargs):
    '''Follow all the starting points at once, each path in its own color'''
    ax = contour_plot(F)
    for c,x0 in zip(colors,starts):
        path, status = newton_trace(x0,solver=solver,**kwargs)
        ax.plot(path[:,0],path[:,1],c=c,marker='.',lw=1.2,zorder=2)  # the path
        ax.scatter(*path[0],c=[c],marker='o',s=25,zorder=3)          # starting point
        if status == 'converged':
            ax.scatter(*path[-1],c=[c],marker='*',s=140,edgecolors='k',lw=.4,zorder=4)
        else:  # mark where the solver gave up
            ax.scatter(*path[-1],c=[c],marker='X',s=80,edgecolors='k',lw=.4,zorder=4)
    ax.scatter([],[],c='grey',marker='o',s=25,label='start')            # legend only
    ax.scatter([],[],c='grey',marker='*',s=140,edgecolors='k',lw=.4,label='converged')
    ax.scatter([],[],c='grey',marker='X',s=80,edgecolors='k',lw=.4,label='gave up')
    ax.legend(loc='upper left',fontsize=8,framealpha=0.8)
    plt.show()

plot_newton_paths()

## Newton finds critical points, not maxima

In [ ]:
for x0 in [(0.55,0.45),(0.55,0.40),(0.20,0.30),(0.75,0.25),
           (0.70,0.50),(0.55,0.65),(0.50,0.50),(0.10,0.80)]:
    try:
        xs, path = newton_path(x0)
        ev = np.linalg.eigvalsh(np.asarray(H(*xs)))  # Hessian at the solution
        kind = 'maximum' if (ev<0).all() else 'minimum' if (ev>0).all() else 'saddle'
        print('from (%4.2f,%4.2f) --> (%6.4f,%6.4f)  F = %5.3f  %-7s  in %2d steps'
              % (*x0,*xs,F(*xs),kind,len(path)-1))
    except RuntimeError as e:
        print('from (%4.2f,%4.2f) --> %s'%(*x0,e))

## Choosing $\lambda$ by step halving

In [ ]:
def newton2_ascent(fun,grad,x0,obj=F,tol=1e-6,maxiter=100,maxhalve=25,callback=None):
    '''Newton method with a step-halving line search on the criterion obj.
    A step is accepted only when obj increases, so the iterations always climb.
    '''
    npfun  = lambda x: np.asarray(fun(x[0],x[1]))
    npgrad = lambda x: np.asarray(grad(x[0],x[1]))
    x0 = np.asarray(x0,dtype=float)
    obj0 = obj(*x0)  # criterion at the current point
    for i in range(maxiter):
        step = np.linalg.solve(npgrad(x0),npfun(x0))  # the full Newton step
        lam = 1.0
        for j in range(maxhalve):  # step-halving line search
            x1 = x0 - lam*step
            obj1 = obj(*x1)
            if obj1 > obj0: break  # uphill, accept this lambda
            lam = lam/2
        else:
            raise RuntimeError('No ascent direction at iteration %d'%i)
        err = np.amax(np.abs(x1-x0))
        if callback != None: callback(iter=i,err=err,x0=x0,x1=x1,lam=lam)
        if err<tol: break
        x0, obj0 = x1, obj1
    else:
        raise RuntimeError('Failed to converge in %d iterations'%maxiter)
    return x1

## The same starting points, with a line search

In [ ]:
plot_newton_paths(newton2_ascent)

center, side = np.array([0.5,0.4]), 0.5  # the small square of starting points
starts = center + np.random.default_rng(5).uniform(-side/2,side/2,size=(25,2))
colors = plt.cm.autumn(np.linspace(0,0.85,len(starts)))

plot_newton_paths(solver=newton2_ascent,starts=starts,colors=colors)